In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.11786820739507675
epoch:  1, loss: 0.07070900499820709
epoch:  2, loss: 0.04901904985308647
epoch:  3, loss: 0.039375875145196915
epoch:  4, loss: 0.03524112328886986
epoch:  5, loss: 0.033515553921461105
epoch:  6, loss: 0.032800402492284775
epoch:  7, loss: 0.03249625489115715
epoch:  8, loss: 0.03235486522316933
epoch:  9, loss: 0.03227778524160385
epoch:  10, loss: 0.03209533542394638
epoch:  11, loss: 0.031941283494234085
epoch:  12, loss: 0.03186345100402832
epoch:  13, loss: 0.03175187483429909
epoch:  14, loss: 0.03157810494303703
epoch:  15, loss: 0.03149612993001938
epoch:  16, loss: 0.03148198127746582
epoch:  17, loss: 0.03127322718501091
epoch:  18, loss: 0.031185273081064224
epoch:  19, loss: 0.031139148399233818
epoch:  20, loss: 0.031014971435070038
epoch:  21, loss: 0.03091573715209961
epoch:  22, loss: 0.03086617961525917
epoch:  23, loss: 0.030764853581786156
epoch:  24, loss: 0.03064741939306259
epoch:  25, loss: 0.030592331662774086
epoch:  26, l

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7399014234542847
Test metrics:  R2 = 0.7265749573707581


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    c1=1e-4,
    c2=0.9,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="wolfe"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.11762111634016037
epoch:  1, loss: 0.1136549711227417
epoch:  2, loss: 0.10987412929534912
epoch:  3, loss: 0.1062697097659111
epoch:  4, loss: 0.10283416509628296
epoch:  5, loss: 0.09956064075231552
epoch:  6, loss: 0.09644025564193726
epoch:  7, loss: 0.09346561133861542
epoch:  8, loss: 0.09063047915697098
epoch:  9, loss: 0.08792907744646072
epoch:  10, loss: 0.08535546064376831
epoch:  11, loss: 0.08290328830480576
epoch:  12, loss: 0.08056623488664627
epoch:  13, loss: 0.07833908498287201
epoch:  14, loss: 0.07621639966964722
epoch:  15, loss: 0.07419292628765106
epoch:  16, loss: 0.07226388901472092
epoch:  17, loss: 0.07042454928159714
epoch:  18, loss: 0.06867111474275589
epoch:  19, loss: 0.0669998750090599
epoch:  20, loss: 0.0654066950082779
epoch:  21, loss: 0.06388814747333527
epoch:  22, loss: 0.06244031712412834
epoch:  23, loss: 0.061059970408678055
epoch:  24, loss: 0.059743862599134445
epoch:  25, loss: 0.058488938957452774
epoch:  26, loss: 0.057

In [8]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6692445874214172
Test metrics:  R2 = 0.6924785375595093


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=10,
    c1=1e-4,
    c2=0.9,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="strong-wolfe"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.08929022401571274
epoch:  1, loss: 0.06536616384983063
epoch:  2, loss: 0.05159974843263626
epoch:  3, loss: 0.04368205368518829
epoch:  4, loss: 0.03913149610161781
epoch:  5, loss: 0.036515846848487854
epoch:  6, loss: 0.03501453623175621
epoch:  7, loss: 0.03415604680776596
epoch:  8, loss: 0.033662497997283936
epoch:  9, loss: 0.033378008753061295
epoch:  10, loss: 0.033213723450899124
epoch:  11, loss: 0.03311891108751297
epoch:  12, loss: 0.033063411712646484
epoch:  13, loss: 0.03303061053156853
epoch:  14, loss: 0.0330108143389225
epoch:  15, loss: 0.03299843147397041
epoch:  16, loss: 0.03298738971352577
epoch:  17, loss: 0.03297238424420357
epoch:  18, loss: 0.032960858196020126
epoch:  19, loss: 0.03294353932142258
epoch:  20, loss: 0.03293543681502342
epoch:  21, loss: 0.032915566116571426
epoch:  22, loss: 0.03290295973420143
epoch:  23, loss: 0.03288803622126579
epoch:  24, loss: 0.032873447984457016
epoch:  25, loss: 0.03286063298583031
epoch:  26, los

In [10]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7585412263870239
Test metrics:  R2 = 0.7421658635139465


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=10,
    c1=1e-4,
    c2=0.9,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="goldstein"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.1869453340768814
epoch:  1, loss: 0.1869453340768814
epoch:  2, loss: 0.1869453340768814
epoch:  3, loss: 0.1869453340768814
epoch:  4, loss: 0.1869453340768814
epoch:  5, loss: 0.1869453340768814
epoch:  6, loss: 0.1869453340768814
epoch:  7, loss: 0.1869453340768814
epoch:  8, loss: 0.1869453340768814
epoch:  9, loss: 0.1869453340768814
epoch:  10, loss: 0.1869453340768814
epoch:  11, loss: 0.1869453340768814
epoch:  12, loss: 0.1869453340768814
epoch:  13, loss: 0.1869453340768814
epoch:  14, loss: 0.1869453340768814
epoch:  15, loss: 0.1869453340768814
epoch:  16, loss: 0.1869453340768814
epoch:  17, loss: 0.1869453340768814
epoch:  18, loss: 0.1869453340768814
epoch:  19, loss: 0.1869453340768814
epoch:  20, loss: 0.1869453340768814
epoch:  21, loss: 0.1869453340768814
epoch:  22, loss: 0.1869453340768814
epoch:  23, loss: 0.1869453340768814
epoch:  24, loss: 0.1869453340768814
epoch:  25, loss: 0.1869453340768814
epoch:  26, loss: 0.1869453340768814
epoch:  27,

In [12]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -33097.015625
Test metrics:  R2 = -32508.078125
